# TPU full profiling — sanity check (JAX device-side trace)

**Paper-fair measurement** of all simulator components:
- dense layers (embedding, layernorm, qkv_proj, rotary_emb, o_proj, gate_up_proj, act_fn, down_proj, final_layernorm)
- per_sequence (lm_head, sampler)
- attention (single-shot prefill + decode sweep — paper batch_sampling grid)

Uses `jax.profiler.trace` + chrome trace JSON parsing — extracts device-side `dur` from `pid=/device:TPU:0` events. Avoids the ~110us host wallclock floor.

**Assumption**: chunked prefill OFF + prefix caching OFF (matches
`studies/tpu_baseline/measure_vllm.py` engine config). Profile is
used by the simulator with `--no-enable-chunked-prefill
--no-enable-prefix-caching`. attention.csv has `kv_prefill = 0` for
every prefill row.

## How to use this notebook

This ipynb is **sanity-check only** — it loads the JAX profiling core
module and runs a small measurement to verify the environment works.

For the **full sweep** (would OOM the kernel due to jit-cache
accumulation), use the subprocess-per-stage runner:

```
./scripts/profile_tpu.sh                                    # defaults: tp=1
TP_LIST="1 2 4 8" ./scripts/profile_tpu.sh                  # multi TP
MODEL=meta-llama/Llama-3.2-3B-Instruct TP=2 ./scripts/profile_tpu.sh
```

## Layout
1. `install` — pip install jax[tpu] etc.
2. `imports` — jax + module
3. **`config` — set MODEL / HARDWARE / TP**
4. **`sanity` — runs `sanity_check(...)` at small input size**


In [ ]:
# Cell 1 — install
import time
from datetime import datetime
_t0 = time.perf_counter()
print(f"[cell 1 start] {datetime.now().isoformat(timespec='seconds')}")

import subprocess, sys
try:
    import jax
    print(f'  jax already installed: {jax.__version__}')
except ImportError:
    print('  installing jax[tpu]...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user',
                            'jax[tpu]',
                            '-f', 'https://storage.googleapis.com/jax-releases/libtpu_releases.html'])
    import jax
for pkg in ('transformers', 'numpy', 'huggingface_hub'):
    try:
        m = __import__(pkg)
        print(f'  {pkg}: OK ({getattr(m, "__version__", "?")})')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', pkg])

print(f"[cell 1 end]   {datetime.now().isoformat(timespec='seconds')}  ({time.perf_counter() - _t0:.2f}s)")

In [ ]:
# Cell 2 — imports + TPU device check
_t0 = time.perf_counter()
print(f"[cell 2 start] {datetime.now().isoformat(timespec='seconds')}")

import os, csv, gzip, json, glob, shutil, tempfile, statistics
from pathlib import Path
from itertools import product

# HF token — set via env OR replace placeholder. Do NOT commit a real token.
os.environ.setdefault('HF_TOKEN', '<your_hf_token>')

import jax
import jax.numpy as jnp
import numpy as np
from tqdm.auto import tqdm
from jax.nn import dot_product_attention as jax_sdpa

print(f'  jax {jax.__version__}, device: {jax.devices()}')
assert any('TPU' in str(d) for d in jax.devices()), 'TPU not detected!'

print(f"[cell 2 end]   {datetime.now().isoformat(timespec='seconds')}  ({time.perf_counter() - _t0:.2f}s)")

In [ ]:
# Cell 3 — load core module (single source of truth, shared with profile_jax.py)
_t0 = time.perf_counter()
print(f"[cell 3 start] {datetime.now().isoformat(timespec='seconds')}")

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from profile_jax_core import (
    resolve_mcfg, build_kernels, measure, sanity_check,
    token_grid, seq_grid, attention_combos,
    _make_dense_input, _make_per_seq_input, _make_attn_input,
)

print('[cell 3] core module loaded — full sweep is in scripts/profile_tpu.sh')
print(f"[cell 3 end]   {datetime.now().isoformat(timespec='seconds')}  ({time.perf_counter() - _t0:.2f}s)")

# SANITY CHECK

For the full sweep use `./scripts/profile_tpu.sh` — see the top markdown.

In [ ]:
# Cell 4 — sanity config (full-sweep config lives in scripts/profile_tpu.sh)
MODEL    = 'meta-llama/Llama-3.2-1B-Instruct'
TP       = 1
WARMUP   = 3
REPEAT   = 10


In [ ]:
# Cell 5 — sanity check
_t0 = time.perf_counter()
print(f"[cell 5 start] {datetime.now().isoformat(timespec='seconds')}")

_mcfg = resolve_mcfg(MODEL, TP)
_kernels = build_kernels(_mcfg)
sanity_check(_mcfg, _kernels, warmup=WARMUP, repeat=REPEAT)

print(f"[cell 5 end]   {datetime.now().isoformat(timespec='seconds')}  ({time.perf_counter() - _t0:.2f}s)")